# Publication-safe notebook

This generalized reference implementation expects a live SparkSession named
`spark` and pseudonymous tables matching `docs/data-contract.md`. It contains no
institution-specific ingestion, credentials, internal paths, data, or results.

Parameter defaults are illustrative and intentionally differ from internal
operating settings. Calibrate them only with authorized development data.


In [ ]:
from pyspark.sql import functions as F
from pyspark import StorageLevel

import os

WRITE_DB = os.environ.get("FRAUD_GRAPH_WRITE_DB", "fraud_graph_demo")
EMP = os.environ.get("FRAUD_GRAPH_PREFIX", "onboarding_fraud")
BASE_TBL = f"{WRITE_DB}.{EMP}_onboarding_base_cohort"
OWN_TBL  = f"{WRITE_DB}.{EMP}_v2_edges_ownership"
EDGES_TXN= f"{WRITE_DB}.{EMP}_v2_edges_transferred"
NODES_ACCT=f"{WRITE_DB}.{EMP}_v2_nodes_account"

BASE_CUST, BASE_MONTH, BASE_FRAUD = "customer_id", "month_period", "final_fraud_flag"

ANCHOR_TRAILING = {
    "202401": ["202310", "202311", "202312"],
    "202402": ["202311", "202312", "202401"],
    "202403": ["202312", "202401", "202402"],
}
KHOP = int(os.environ.get("FRAUD_GRAPH_KHOP", "2"))
EXPANSION_MAX_DEGREE = int(os.environ.get("FRAUD_GRAPH_EXPANSION_MAX_DEGREE", "100"))
DROP_HUB             = True
MAX_NODES_WARN = int(os.environ.get("FRAUD_GRAPH_MAX_NODES_WARN", "1000000"))

SEED_MATURITY_LAG = int(os.environ.get("FRAUD_GRAPH_SEED_MATURITY_LAG", "2"))

def _ym_minus(ym, k):
    t = int(str(ym)[:4]) * 12 + (int(str(ym)[4:6]) - 1) - int(k)
    return "%04d%02d" % (t // 12, t % 12 + 1)

OUT_NIDX = f"{WRITE_DB}.{EMP}_v2_subgraph_node_index"
OUT_EIDX = f"{WRITE_DB}.{EMP}_v2_subgraph_edges_idx"

def ACC_KEY(c): return F.upper(F.trim(F.col(c).cast("string")))
def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

base = spark.table(BASE_TBL)
own  = spark.table(OWN_TBL).select(F.col("src").alias("cust"), F.col("dst").alias("acct"))
nacc = spark.table(NODES_ACCT).select("node_id", "account_network_type", "is_hub")
print("anchors:", list(ANCHOR_TRAILING.keys()), "| KHOP:", KHOP, "| degree cap:", EXPANSION_MAX_DEGREE)


In [ ]:
def seed_cand_accounts(anchor):
    b = base.select(ACC_KEY(BASE_CUST).alias("cust"),
                    F.col(BASE_MONTH).cast("string").alias("m"),
                    F.col(BASE_FRAUD).cast("int").alias("ff")).where(F.col("cust").isNotNull())

    cand_cust = (b.where(F.col("m") == anchor).groupBy("cust").agg(F.max("ff").alias("target_fraud")))

    seed_cutoff = _ym_minus(anchor, SEED_MATURITY_LAG)
    seed_cust = (b.where((F.col("m") < seed_cutoff) & (F.col("ff") == 1)).select("cust").distinct())

    cand_acc = (cand_cust.join(own, "cust").select(F.col("acct").alias("id"), "target_fraud")
                .groupBy("id").agg(F.max("target_fraud").alias("target_fraud"))
                .withColumn("is_candidate", F.lit(1)))
    seed_acc = (seed_cust.join(own, "cust").select(F.col("acct").alias("id")).distinct()
                .withColumn("is_known_bad", F.lit(1)))
    return seed_acc, cand_acc


In [ ]:
def build_subgraph(anchor):
    months = ANCHOR_TRAILING[anchor]
    seed_acc, cand_acc = seed_cand_accounts(anchor)

    e = spark.table(EDGES_TXN).where(F.col("txn_month").isin(*months))\
            .select("src", "dst", "sum_amount", "txn_count", F.col("txn_month").cast("int").alias("tm"))
    if DROP_HUB:
        hubs = nacc.where(F.col("is_hub") == 1).select(F.col("node_id").alias("dst"))
        e = e.join(hubs, "dst", "left_anti")
    e = e.where(F.col("src") != F.col("dst")).persist(StorageLevel.DISK_ONLY)

    de = e.select("src", "dst", "tm").distinct().persist(StorageLevel.DISK_ONLY)

    adj = (e.select("src", "dst").union(e.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
           .distinct().persist(StorageLevel.DISK_ONLY))
    udeg = adj.groupBy("src").agg(F.count("*").alias("d")).select(F.col("src").alias("id"), "d")

    roots = (seed_acc.select("id").union(cand_acc.select("id")).distinct()).persist(StorageLevel.DISK_ONLY)
    transit = (udeg.where(F.col("d") <= EXPANSION_MAX_DEGREE).select("id")
               .union(roots).distinct()).persist(StorageLevel.DISK_ONLY)

    frontier = roots.withColumn("arr", F.lit(0))
    visited = roots
    for _h in range(KHOP):
        expandable = frontier.join(transit, "id", "inner")
        nbr = (de.join(expandable.select(F.col("id").alias("src"), "arr"), "src")
               .where(F.col("tm") >= F.col("arr"))
               .select(F.col("dst").alias("id"), F.col("tm").alias("arr")))
        nbr = nbr.groupBy("id").agg(F.min("arr").alias("arr"))
        frontier = nbr.join(visited, "id", "left_anti").persist(StorageLevel.DISK_ONLY)
        if frontier.limit(1).count() == 0:
            break
        visited = visited.union(frontier.select("id")).distinct().persist(StorageLevel.DISK_ONLY)

    reached = visited.select("id").distinct()

    n = reached.count()
    idx = spark.createDataFrame(reached.rdd.map(lambda r: r[0]).zipWithIndex(), ["id", "idx"])
    node_index = (idx
                  .join(seed_acc, "id", "left").join(cand_acc, "id", "left")
                  .join(nacc.withColumnRenamed("node_id", "id"), "id", "left")
                  .fillna({"is_known_bad": 0, "is_candidate": 0, "target_fraud": 0, "is_hub": 0})
                  .withColumn("anchor_month", F.lit(anchor)))

    cand_edges = (e.join(idx.select(F.col("id").alias("src"), F.col("idx").alias("src_idx")), "src")
                  .join(idx.select(F.col("id").alias("dst"), F.col("idx").alias("dst_idx")), "dst")
                  .select(F.lit(anchor).alias("anchor_month"), "src_idx", "dst_idx",
                          F.log1p(F.col("sum_amount")).alias("weight"), "txn_count", "sum_amount"))

    e.unpersist(); de.unpersist(); adj.unpersist(); roots.unpersist(); transit.unpersist()
    return node_index, cand_edges, n


In [ ]:
from functools import reduce
ni_frames, ce_frames = [], []
for a in ANCHOR_TRAILING.keys():
    print("=== building subgraph for", a, "===")
    ni, ce, n = build_subgraph(a)
    print("   subgraph nodes:", n, ("  *** > warn threshold, consider lowering KHOP/degree ***" if n > MAX_NODES_WARN else ""))
    ni_frames.append(ni); ce_frames.append(ce)

node_index = reduce(lambda x, y: x.unionByName(y), ni_frames)
cand_edges = reduce(lambda x, y: x.unionByName(y), ce_frames)
save_overwrite(node_index, OUT_NIDX)
save_overwrite(cand_edges, OUT_EIDX)
print("saved:", OUT_NIDX, "and", OUT_EIDX)


In [ ]:
ni = spark.table(OUT_NIDX); ce = spark.table(OUT_EIDX)
qc_ok = True
for a in ANCHOR_TRAILING.keys():
    nia = ni.where(F.col("anchor_month") == a)
    cea = ce.where(F.col("anchor_month") == a)
    nodes = nia.count(); edges = cea.count()
    seeds = nia.where(F.col("is_known_bad") == 1).count()
    cands = nia.where(F.col("is_candidate") == 1).count()
    badc  = nia.where((F.col("is_candidate") == 1) & (F.col("target_fraud") == 1)).count()

    cidx = nia.where(F.col("is_candidate") == 1).select("idx")
    ep = (cea.select(F.col("src_idx").alias("idx")).union(cea.select(F.col("dst_idx").alias("idx"))).distinct())
    conn = cidx.join(ep, "idx", "inner").count()
    st = nia.agg(F.min("idx").alias("mn"), F.max("idx").alias("mx"), F.count("*").alias("n")).first()
    contiguous = (st["mn"] == 0 and st["mx"] == st["n"] - 1)
    bad_edge = cea.where((F.col("src_idx") >= nodes) | (F.col("dst_idx") >= nodes)).count()
    ok = contiguous and bad_edge == 0
    qc_ok = qc_ok and ok
    print("[%s] nodes %d | edges %d | seeds %d | candidates %d (bad %d) | cand-connected %d | idx_contig %s | bad_edge %d | OK=%s" % (
        a, nodes, edges, seeds, cands, badc, conn, contiguous, bad_edge, ok))


In [ ]:
if qc_ok:
    print("NOTEBOOK 03 (v2) COMPLETE — seed-anchored k-hop subgraph built and integer-indexed for downstream graph feature stages.")
    print("Outputs:", OUT_NIDX, "(nodes + idx + is_known_bad/is_candidate/target_fraud/net_type)")
    print("        ", OUT_EIDX, "(src_idx, dst_idx, weight, txn_count -> indexed graph edges)")
    print("Next: notebook 04 — compute graph features for candidates and validate against point-in-time labels.")
else:
    print("NOTEBOOK 03 (v2) NOT done — a per-anchor QC failed; fix before notebook 04.")
print("Review the per-anchor QC line (nodes/edges/seeds/candidates/connectivity/OK).")
